# Context, roles and Large Language Models

## What are Large Language Models (LLMs) really?

LLMs are basically software trained on huge amounts of text to predict the most probable next word given a starting text, e.g. if we feed it "It was a dark and" it will likely return "stormy". We can then append that response to the original text and repeat the process. If you guess that the next response will be "night" you are probably right. 

```
"It was a dark and" -> "stormy"
"It was a dark and stormy" -> "night"
"It was a dark and stormy night" -> ...
...

```

However, predicting a-word-at-time is not how we normally use LLMs, as they are wrapped up in various interfaces that hides this basic fact as well as making them a lot more useful, e.g. by producing long answers. That chatbots like ChatGPT and others will produce a fairy tale if prompted with "Once upon a time" might not come as a surprise. 

# Completion

We use LLMs wrapped up in the sofware Ollama, which acts as an interface in between a "raw" LLM and a chatbot. We can use the simplest possible API call `generate` to see what is going on:

In [ ]:
!pip -q install ollama

In [ ]:
from ollama import Client

OLLAMA_HOST = 'http://10.129.20.4:9090'
OLLAMA_MODEL = 'llama3.1:latest'

client = Client(host=OLLAMA_HOST)

In [ ]:
prompt = 'It was a dark and…'
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

Hopefully the printed answer is at least starting with '...stormy night!', but as the output is not deterministic there is no guarantee. Re-run the above cell to see different answers. In fact, that is true for all the LLM tutorials: Re-running cells will produce different results most of the time.

## Adding context

By adding _context_, i.e. more text to the prompt we can **steer** (not control) the output from the LLM:

In [ ]:
prompt = 'You are Super Mario. It was a dark and…'
reply = client.generate(
    OLLAMA_MODEL,
    prompt=prompt,
    stream = False,
)
print(reply.response)

## Chat

It is important to remember that the LLM has no memory of previous prompts, which is why they are almost always used via a `chat` API where the conversation as a whole is fed to the LLM **every time** new input is provided. To keep things organized, the chat history is kept in a list (named `messages` below), with each entry tagged with the _role_ of the part providing the input. The human is commonly tagged with `user`. When the reply comes back it is appended to the list of messages, this time tagged as `assistant`.

In [ ]:
messages = [{'role': 'user', 'content': 'Who are you?'}]

reply = client.chat(
    OLLAMA_MODEL,
    messages=messages,
    stream = False,
)
messages.append(reply.message)

To help us examine the message list we'll add a helper function:

In [ ]:
import json
from IPython import display

def show_messages(messages):
    """Helper function to view messages in a nice format"""
    s = json.loads(json.dumps(messages, default=dict))
    return display.JSON(s, root="Message history")

Running it will present the list as entries that can be shown/hidden by clicking the triangles. Try it. For now, focus on the `role` and `content` fields, and ignore the rest.

In [ ]:
show_messages(messages)

### The system role

Most LLMs support a `system` role to set the fundamental, overarching context:

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are Super Mario.'},
    {'role': 'user', 'content': 'Who are you?'}
]

reply = client.chat(
    OLLAMA_MODEL,
    messages=messages,
    stream = False,
)
messages.append(reply.message)
show_messages(messages)

Here we can see in the `assistant`s reply (entry #2) that the same question (entry #1) as above is answered quite differently based on the system message (entry #0).

### The tool role

Some models are capable of requesting help to solve particular tasks by asking for _tool_ help (see LLM-tool-calling tutorial) but the end effect is just adding more context to the message history. We can fake tool input to the context by manually injecting some piece of information that constitutes a grounding in facts for the LLM. This time the entry is tagged `tool` and it has an additional field stating the name of tool providing the content (here we'll use the made up name 'time' to comply with API requirements). 

In [ ]:
messages.extend([
    {'role': 'tool', 'content': '11 AM', 'name': 'time'},
    {'role': 'user', 'content': 'What time is it?'}
])

reply = client.chat(
    OLLAMA_MODEL,
    messages=messages,
    stream = False,
)
messages.append(reply.message)
show_messages(messages)

To help in detangling the message list above, as you walk through the entries, you can consult the diagram below. What is actually sent to the LLM is indicated by the dotted arrows.

```mermaid
sequenceDiagram
    participant LLM as LLM ('assistant')
    participant messages@{ "type" : "database" }
    participant TOOL as 'tool'
    actor USER as 'user'

    USER->>messages: Who are you?
    messages-->>LLM: (0) You are Super Mario, (1) Who are you?
    LLM->>messages: It's-a me, Mario...
    TOOL->>messages: 11 AM
    USER->>messages: What time is it?
    messages-->>LLM: (0) You are Super Mario, (1) Who are you?, (2) It's-a me, Mario..., (3) 11 AM, (4) What time is it?
    LLM->>messages: It's-a 11:00 AM in the Mushroom Kingdom...
```

## Summary

So, to summarize, as we conversate with an LLM the context is built up (on the client side) by keeping a record of the transactions so far (in `messages`). To make more sense, each entry is tagged with a role to mark its origin. The most obvious roles are `user` and `assistant` as the are necessary participants in a conversation. The `system` role is supported by most LLMs, and role `tool` is only available in models trained to ask for help (a.k.a tool-calling models). To find out what roles, and their exact names, are supported by a particular model you can consult their _model card_ (see e.g. ). To make things a little more confusing, Ollama tries to be helpful by automagically mapping the "standard" names `system`, `user`, `assistant`, and `tool` to any "non-standard" names used by a particular model _for that role_. In short, you can be fairly certian that using the names `system`, `user`, `assistant`, and `tool` with Ollama will work, as long as that role is supported by the model (the tool role in particular is only supported by certain models).

The list of roles is not static in this quickly developing field, but the roles outlined here are not likely to go away any time soon.